In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
from google.colab import userdata
import os

os.environ["HUGGINGFACEHUB_API_TOKEN"] = userdata.get("HUGGINGFACEHUB_API_TOKEN")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [ ]:
# 1 — Installation des dépendances
%pip install -q \
    transformers \
    sentence-transformers \
    faiss-cpu \
    pymupdf \
    beautifulsoup4 \
    accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 72.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 78.0 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [ ]:
! pip install gradio

In [ ]:
# Avec Reranker
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)


2026-01-14 13:58:05.375690: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768399085.811992      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768399085.937530      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768399086.939592      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768399086.939631      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768399086.939634      55 computation_placer.cc:177] computation placer alr

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [ ]:
# 2 — Imports & paramètres globaux
import os
import fitz
import faiss
import torch
import numpy as np
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline


In [ ]:
# =========================
# PARAMÈTRES
# =========================

DATA_DIR = "/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Wahib B/cleaned_json_full/cleaned_json_full"





MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

CHUNK_SIZE = 250

OVERLAP = 30

TOP_K = 4

SIMILARITY_THRESHOLD = 0.35

MAX_CONTEXT_CHARS = 2000
MAX_NEW_TOKENS = 150


In [ ]:
import os, json
from pathlib import Path

# =========================
# OUTILS
# =========================
def safe_get_text(obj):
    """
    Extrait du texte depuis différents formats JSON.
    - Si JSON concours structuré (keys concours_num + postes), on le "linéarise" en texte.
    - Sinon, fallback générique (dict/list/str).
    """
    if obj is None:
        return ""

    # --- CAS SPECIAL: JSON concours structuré ---
    if isinstance(obj, dict) and ("concours_num" in obj) and ("postes" in obj):
        lines = []

        # Métadonnées concours
        for k, label in [
            ("concours_label", "Concours"),
            ("concours_num", "Numéro"),
            ("bap", "BAP"),
            ("grade", "Grade"),
            ("emploi_type", "Emploi-type"),
            ("nb_postes", "Nombre de postes"),
            ("nb_postes_detectes", "Nombre de postes détectés"),
            ("source_file", "Source"),
        ]:
            v = obj.get(k, None)
            if v is not None and str(v).strip() != "":
                lines.append(f"{label} : {v}")

        # Détails postes
        postes = obj.get("postes", [])
        if isinstance(postes, list) and postes:
            for i, p in enumerate(postes, start=1):
                if not isinstance(p, dict):
                    continue
                lines.append(f"\nPOSTE {i} :")
                # On prend toutes les infos du poste, même si les clés varient
                for pk, pv in p.items():
                    if pv is None:
                        continue
                    # listes -> concat
                    if isinstance(pv, list):
                        pv = " ; ".join(str(x) for x in pv if str(x).strip())
                    # dict -> string simple
                    elif isinstance(pv, dict):
                        pv = " ; ".join(f"{a}={b}" for a, b in pv.items() if str(b).strip())
                    else:
                        pv = str(pv)

                    pv = pv.strip()
                    if pv:
                        lines.append(f"- {pk} : {pv}")

        return "\n".join(lines).strip()

    # --- FALLBACK GENERIQUE ---
    if isinstance(obj, str):
        return obj.strip()

    if isinstance(obj, list):
        parts = []
        for it in obj:
            t = safe_get_text(it)
            if t:
                parts.append(t)
        return "\n".join(parts).strip()

    if isinstance(obj, dict):
        # si jamais un JSON "texte" existe
        for k in ["text", "content", "clean_text", "raw_text", "body", "page_content"]:
            if k in obj and isinstance(obj[k], str) and obj[k].strip():
                return obj[k].strip()

        parts = []
        for v in obj.values():
            t = safe_get_text(v)
            if t:
                parts.append(t)
        return "\n".join(parts).strip()

    return ""


def chunk_text(text, chunk_size=250, overlap=30):
    """
    Chunk par mots, avec overlap.
    """
    words = text.split()
    step = max(1, chunk_size - overlap)
    for i in range(0, len(words), step):
        chunk = " ".join(words[i:i + chunk_size]).strip()
        if chunk:
            yield chunk


# =========================
# 1) CHECK DOSSIER
# =========================
print("DATA_DIR exists:", os.path.isdir(DATA_DIR))
print("Exemples fichiers:", os.listdir(DATA_DIR)[:10])

# =========================
# 2) CHARGEMENT JSON
# =========================
json_files = sorted([p for p in Path(DATA_DIR).glob("*.json")])
print("Nb fichiers JSON:", len(json_files))

documents = []
skipped = 0

for fp in json_files:
    try:
        with open(fp, "r", encoding="utf-8") as f:
            obj = json.load(f)

        full_text = safe_get_text(obj)
        if not full_text:
            skipped += 1
            continue

        # Métadonnées source
        source = fp.name

        # Déduire la page depuis le nom du fichier (ex: page_009.json -> 9)
        page = "N/A"
        if fp.stem.startswith("page_"):
            try:
                page = int(fp.stem.split("_")[1])
            except:
                page = "N/A"

        # Chunk
        for chunk in chunk_text(full_text, CHUNK_SIZE, OVERLAP):
            documents.append({
                "text": chunk,
                "source": source,
                "page": page
            })

    except Exception as e:
        skipped += 1
        print(f"⚠️ Erreur fichier {fp.name}: {e}")


print(f"✅ Chunks créés: {len(documents)}")
print(f"⚠️ Fichiers ignorés/erreurs: {skipped}")

# =========================
# 3) APERÇU
# =========================
if documents:
    print("\n--- Exemple chunk ---")
    print("SOURCE:", documents[0]["source"])
    print(documents[0]["text"][:800])


DATA_DIR exists: True
Exemples fichiers: ['page_003.json', 'page_112.json', 'page_048.json', 'page_073.json', 'page_025.json', 'page_074.json', 'page_096.json', 'page_016.json', 'page_121.json', 'page_089.json']
Nb fichiers JSON: 127
✅ Chunks créés: 625
⚠️ Fichiers ignorés/erreurs: 0

--- Exemple chunk ---
SOURCE: guide_candidat_2025.json
CNRS – Guide candidat(e) 2025 (IT) Guide candidat 2025.pdf CONCOURS EXTERNES DES PERSONNELS INGÉNIEURS ET TECHNICIENS Le guide du candidat et de la candidate Edition 2025 Direction de la publication : Antoine Petit Direction de la rédaction : Hélène Maury Direction adjointe de la rédaction : Christiane Ename – Laetitia Navarro -Service recrutement et intégration (SeRI) Autrices : Dominique Marx - Emilie Faure - Nathalie Nioucel Mai 2025 5 - 6 Pourquoi candidater ? 7 - 8 Le choix des concours 9 - 10 L’inscription 11 Comment concourir ? 12 Les conditions pour concourir 13 - 14 Le déroulement des concours 15-16 Les épreuves 17 La publication des résultat

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3 — Détection GPU automatique
def can_use_gpu(min_free_gb=4):
    if not torch.cuda.is_available():
        return False
    free, total = torch.cuda.mem_get_info()
    return free / (1024**3) >= min_free_gb

USE_GPU = can_use_gpu()
print(f"🔍 Mode sélectionné : {'GPU' if USE_GPU else 'CPU'}")


🔍 Mode sélectionné : GPU


In [ ]:
import json
from pathlib import Path

def safe_get_text(obj):
    if isinstance(obj, dict):
        for k in ["text", "content", "clean_text", "body"]:
            if k in obj and isinstance(obj[k], str):
                return obj[k].strip()
        for v in obj.values():
            t = safe_get_text(v)
            if t:
                return t
    if isinstance(obj, list):
        return "\n".join(filter(None, (safe_get_text(x) for x in obj)))
    if isinstance(obj, str):
        return obj.strip()
    return ""


def chunk_text(text):
    words = text.split()
    step = max(1, CHUNK_SIZE - OVERLAP)
    for i in range(0, len(words), step):
        yield " ".join(words[i:i + CHUNK_SIZE])


documents = []

json_files = list(Path(DATA_DIR).glob("*.json"))
print("📁 Fichiers JSON trouvés :", len(json_files))

for fp in json_files:
    with open(fp, "r", encoding="utf-8") as f:
        obj = json.load(f)

    full_text = safe_get_text(obj)
    if not full_text.strip():
        continue

    for chunk in chunk_text(full_text):
        documents.append({
            "text": chunk,
            "source": fp.name
        })

print(f"📄 Documents indexés : {len(documents)} chunks")


📁 Fichiers JSON trouvés : 127
📄 Documents indexés : 127 chunks


In [ ]:
# 5 — Embeddings & FAISS
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

assert len(documents) > 0, "documents est vide — vérifie le chargement/chunking avant FAISS."

embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cuda" if USE_GPU else "cpu"
)

texts = [d["text"] for d in documents]

embeddings = embedder.encode(
    texts,
    normalize_embeddings=True,   # => cosine similarity si IndexFlatIP
    batch_size=16,
    show_progress_bar=True
)

embeddings = np.asarray(embeddings, dtype="float32")
embeddings = np.ascontiguousarray(embeddings)  # FAISS aime le contigu

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

print(f"✅ FAISS prêt — {index.ntotal} vecteurs, dim={dim}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

✅ FAISS prêt — 127 vecteurs, dim=384


In [ ]:
# 6 — Retrieval avec Reranker
import numpy as np

def retrieve(question):
    # 1) Retrieval large
    q_emb = embedder.encode([question], normalize_embeddings=True)
    q_emb = np.asarray(q_emb, dtype="float32")
    q_emb = np.ascontiguousarray(q_emb)

    scores, indices = index.search(q_emb, TOP_K * 3)

    candidates = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        # seuil "soft" (optionnel) : garde-le bas pour ne pas perdre des bons passages
        if score >= SIMILARITY_THRESHOLD:
            candidates.append({**documents[idx], "retrieval_score": float(score)})

    if not candidates:
        return []

    # 2) Reranking précis
    pairs = [(question, c["text"]) for c in candidates]
    rerank_scores = reranker.predict(pairs)

    reranked = sorted(
        zip(rerank_scores, candidates),
        key=lambda x: x[0],
        reverse=True
    )

    # 3) On garde les meilleurs
    results = []
    for s, c in reranked[:TOP_K]:
        c2 = dict(c)
        c2["rerank_score"] = float(s)
        results.append(c2)

    return results


In [ ]:
# 7 — Chargement du modèle Mistral (fix accelerate/pipeline)
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if USE_GPU:
    # GPU (accelerate device_map auto)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True
    )

    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        return_full_text=False,
        pad_token_id=tokenizer.eos_token_id
    )

else:
    # CPU (pas accelerate)
    os.environ["CUDA_VISIBLE_DEVICES"] = ""
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map={"": "cpu"},
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True
    )
    torch.set_num_threads(4)

    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        device=-1,  # optionnel, mais OK sur CPU
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        return_full_text=False,
        pad_token_id=tokenizer.eos_token_id
    )

print("✅ Modèle + pipeline prêts :", MODEL_NAME, "| GPU" if USE_GPU else "| CPU")


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

...

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✅ Modèle + pipeline prêts : mistralai/Mistral-7B-Instruct-v0.2 | GPU


In [ ]:
# 8 — Prompt CNRS STRICT (anti-hallucination)
def build_prompt(question, contexts):
    def fmt_source(c):
        page = c.get("page", None)
        if page is None or page == "N/A":
            return f"{c.get('source', 'inconnu')}"
        return f"{c.get('source', 'inconnu')} | page {page}"

    # Contexte (on limite chaque extrait + on limite le total)
    blocks = []
    total_chars = 0

    for c in contexts:
        excerpt = (c.get("text", "") or "").strip()
        if not excerpt:
            continue

        excerpt = excerpt[:MAX_CONTEXT_CHARS]
        block = f"- {fmt_source(c)}\n  {excerpt}"

        if total_chars + len(block) > MAX_CONTEXT_CHARS * max(1, TOP_K):
            break

        blocks.append(block)
        total_chars += len(block)

    sources_block = "\n".join(blocks) if blocks else "- (aucun contexte)"

    return f"""<s>[INST] Tu es un agent officiel d'information sur les concours ingénieur du CNRS.

RÈGLES ABSOLUES :
- Tu utilises EXCLUSIVEMENT les sources ci-dessous.
- Tu ne déduis rien.
- Tu ne complètes rien.
- Tu ne poses pas de nouvelle question.
- Tu ne réponds qu'UNE SEULE FOIS.
- Tu réponds uniquement en français.

FORMAT DE SORTIE OBLIGATOIRE :

RÉPONSE :
<réponse factuelle>

SOURCES :
- <fichier> | page <numéro>

SI l'information n'est PAS clairement présente, répond EXACTEMENT :

RÉPONSE :
Je ne dispose pas de cette information dans les documents de référence.

SOURCES :
Aucune

SOURCES DISPONIBLES :
{sources_block}

QUESTION :
{question} [/INST]
"""


In [ ]:
# Bloquer / nettoyer la sortie du modèle
def clean_output(text):
    stop_markers = [
        "\nQUESTION :",
        "\n❓",
        "\n[INST]",
        "\nSOURCES DISPONIBLES",
        "\nSOURCES :",
        "\nSOURCE :",
    ]

    for marker in stop_markers:
        if marker in text:
            text = text.split(marker)[0]

    return text.strip()


In [ ]:
# 9 — Fonction answer() finale
REFUS = "Je ne dispose pas de cette information dans les documents de référence."

def answer(question):
    contexts = retrieve(question)

    # Si rien trouvé
    if not contexts:
        return f"RÉPONSE :\n{REFUS}\n\nSOURCES :\nAucune"

    prompt = build_prompt(question, contexts)

    # Génération
    gen = pipe(prompt)[0]["generated_text"]
    output = clean_output(gen)

    # Si le modèle refuse (ou ne respecte pas le format), on force le refus
    if (REFUS in output) or ("RÉPONSE :" not in output):
        return f"RÉPONSE :\n{REFUS}\n\nSOURCES :\nAucune"

    return output


In [ ]:
# 10 — Mode interactif (démo)
print("\n🤖 Agent CNRS prêt. Tape 'quitter' pour quitter.\n")

while True:
    q = input("❓ Question : ").strip()

    if not q:
        continue

    if q.lower() in {"quitter", "quit", "exit"}:
        break

    print("\n" + answer(q))
    print("\n" + "-" * 60)



🤖 Agent CNRS prêt. Tape 'quitter' pour quitter.

❓ Question : Quels sont les avantages à travailler comme ingénieur au CNRS en termes de carrière et de conditions de travail ?

RÉPONSE :
Les avantages en matière de carrière et de conditions de travail pour travailler comme ingénieur au CNRS sont détaillés dans le document "CNRS Carrières – Vos avantages" (page 2 et suivantes).

------------------------------------------------------------
❓ Question : exit


# Metrique

# Fidélité / Ancrage
La réponse est-elle soutenue par le contexte récupéré ?

LLM-as-Judge :

In [ ]:
import torch
from transformers import pipeline

judge_pipe = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    device=0 if torch.cuda.is_available() else -1
)



config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


In [ ]:
import pandas as pd

data = pd.read_csv("/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Mohamed-Taha Belhaj - Analyse/jeu_test.csv")

data.head()


,ID,Question,expected_answer,category,must_refuse,must_cite_source,expected_sources
0,1,Combien de postes sont disponibles pour le con...,Le concours comprend trois postes distincts :\...,concours_info,0.0,1.0,page_001.html
1,2,Est ce que le concours Experte ou expert en in...,"Pour le concours n°64, le grade mentionné est ...",orientation,0.0,1.0,Guide candidat 2025.pdf
2,3,Quelles sont les responsabilités de l'ingénieu...,L'agent développera des outils numériques avan...,concours_info,0.0,1.0,page_064.html
3,4,Quel salaire je peux prétendre en fin de carri...,Le concours numéro 64 fait référence à un pos...,career_general,0.0,1.0,Guide candidat 2025.pdf
4,5,Est ce que le lieu du concours Experte ou expe...,La fiche du concours indique que le lieu de tr...,process_general,1.0,0.0,NaN


In [ ]:
# Fidelité
def evaluate_faithfulness(answer, contexts, judge_pipe, max_context_chars=2000):
    context_text = " ".join(c.get("text", "") for c in contexts)[:max_context_chars]

    prompt = f"""
Cette réponse est-elle fidèle au contexte ?
Réponds uniquement par OUI ou NON.

Contexte :
{context_text}

Réponse :
{answer}

La réponse est-elle soutenue par le contexte ?
"""

    out = judge_pipe(prompt, max_new_tokens=5, do_sample=False)[0]["generated_text"].lower().strip()
    return 1 if out.startswith("oui") else 0


def extract_answer_only(ans_text):
    if "RÉPONSE :" in ans_text:
        ans_text = ans_text.split("RÉPONSE :")[1]
    if "SOURCES :" in ans_text:
        ans_text = ans_text.split("SOURCES :")[0]
    return ans_text.strip()


In [ ]:
import numpy as np

faithfulness_scores = []
all_results = []

for idx, row in data.iterrows():
    q = row["Question"]

    contexts = retrieve(q)
    ans = answer(q)

    ans_only = extract_answer_only(ans)

    score = evaluate_faithfulness(
        answer=ans_only,
        contexts=contexts,
        judge_pipe=judge_pipe
    )

    faithfulness_scores.append(score)

    all_results.append({
        "ID": row.get("ID", idx),
        "Question": q,
        "Faithfulness": score
    })

print("Fidélité moyenne :", np.mean(faithfulness_scores))
print("Fidélité :", faithfulness_scores)


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Fidélité moyenne : 0.0
Fidélité : [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


# Pertinence de la Réponse
***La réponse aborde-t-elle la question ?***

In [ ]:
import re
import torch

def evaluate_answer_relevancy(question, answer, judge_pipe, max_new_tokens=10):

    prompt = f"""
Donne une note de 1 à 5 uniquement (un seul chiffre).

Question :
{question}

Réponse :
{answer}

Pertinence (1 = hors sujet, 5 = répond parfaitement) :
"""

    out = judge_pipe(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )[0]["generated_text"].strip()

    match = re.search(r"\b([1-5])\b", out)
    if not match:
        return 0.0

    score = int(match.group(1))
    return score / 5.0


In [ ]:
import numpy as np
import pandas as pd

relevancy_scores = []
all_results = []

for idx, row in data.iterrows():
    q = row["Question"]

    ans = answer(q)
    ans_only = extract_answer_only(ans)

    score = evaluate_answer_relevancy(
        question=q,
        answer=ans_only,
        judge_pipe=judge_pipe
    )

    relevancy_scores.append(score)

    all_results.append({
        "ID": row.get("ID", idx),
        "Question": q,
        "Relevancy": score
    })

print("Pertinence moyenne :", np.mean(relevancy_scores))
print("Pertinence :", relevancy_scores)


Pertinence moyenne : 0.020833333333333332
Pertinence : [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.2, 0.2, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.2, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.2, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.2, 0.0, 0.0, 0.0, 0.0]


# Précision du Contexte
Le contexte récupéré est-il pertinent ?

In [ ]:
def evaluate_context_precision(question, contexts, judge_pipe, max_chunk_chars=800):
    """
    Évalue si les chunks récupérés sont pertinents pour la question.
    Retourne un score entre 0 et 1.
    """
    if not contexts:
        return 0.0

    relevant = 0

    for c in contexts:
        chunk_text = (c.get("text", "") or "")[:max_chunk_chars]

        prompt = f"""
Réponds uniquement par OUI ou NON.

Question :
{question}

Contexte :
{chunk_text}

Ce contexte est-il pertinent pour répondre à la question ?
"""

        out = judge_pipe(
            prompt,
            max_new_tokens=3,
            do_sample=False
        )[0]["generated_text"].lower().strip()

        if out.startswith("oui"):
            relevant += 1

    return relevant / len(contexts)


In [ ]:
import numpy as np
import pandas as pd

context_precision_scores = []
all_results = []

for idx, row in data.iterrows():
    q = row["Question"]

    contexts = retrieve(q)

    score = evaluate_context_precision(
        question=q,
        contexts=contexts,
        judge_pipe=judge_pipe
    )

    context_precision_scores.append(score)

    all_results.append({
        "ID": row.get("ID", idx),
        "Question": q,
        "Context_Precision": score
    })

print("📊 Précision moyenne du contexte :", np.mean(context_precision_scores))
print("📊 Précision du contexte :", context_precision_scores)


📊 Précision moyenne du contexte : 0.0
📊 Précision du contexte : [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


# Rappel du Contexte
Toutes les informations nécessaires sont-elles dans le contexte récupéré ?

In [ ]:
def evaluate_context_recall(ground_truth_answer, contexts, judge_pipe, max_context_chars=1500):
    """
    Vérifie si le contexte récupéré contient toutes les infos nécessaires
    pour produire la réponse attendue.
    Retourne 1 ou 0.
    """

    if not contexts:
        return 0

    context_text = " ".join(
        (c.get("text", "") or "")[:400] for c in contexts
    )[:max_context_chars]

    prompt = f"""
Réponds uniquement par OUI ou NON.

Règle stricte :
- OUI = toutes les informations essentielles sont présentes dans le contexte
- NON = s'il manque au moins une information essentielle

Contexte :
{context_text}

Réponse attendue :
{ground_truth_answer}

Toutes les informations nécessaires pour produire EXACTEMENT cette réponse
sont-elles présentes dans le contexte ?
"""

    out = judge_pipe(
        prompt,
        max_new_tokens=3,
        do_sample=False
    )[0]["generated_text"].lower().strip()

    return 1 if out.startswith("oui") else 0


In [ ]:
import numpy as np
import pandas as pd

context_recall_scores = []
all_results = []

for idx, row in data.iterrows():
    q = row["Question"]
    gt = row["expected_answer"]

    contexts = retrieve(q)

    score = evaluate_context_recall(
        ground_truth_answer=gt,
        contexts=contexts,
        judge_pipe=judge_pipe
    )

    context_recall_scores.append(score)

    all_results.append({
        "ID": row.get("ID", idx),
        "Question": q,
        "Context_Recall": score
    })

print("📊 Rappel moyen du contexte :", np.mean(context_recall_scores))
print("📊 Rappel du contexte :", context_recall_scores)


📊 Rappel moyen du contexte : 0.0
📊 Rappel du contexte : [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


# Fonction Refusal Accuracy

In [ ]:
import re

REFUSAL_PATTERNS = [
    r"je ne dispose pas",
    r"je n'ai pas (?:cette|ces) information",
    r"information .*pas .*dans les documents",
    r"aucune (?:source|information)",
    r"je ne peux pas répondre",
]

def is_refusal(text: str) -> bool:
    if not text:
        return True
    t = text.lower().strip()
    return any(re.search(p, t) for p in REFUSAL_PATTERNS)

def refusal_accuracy(answers, ground_truths):
    if len(answers) == 0:
        return 0.0

    correct = 0
    for ans, gt in zip(answers, ground_truths):
        gt_refusal = is_refusal(gt)
        ans_refusal = is_refusal(ans)

        if gt_refusal == ans_refusal:
            correct += 1

    return correct / len(answers)


In [ ]:
import numpy as np
import pandas as pd

all_results = []
errors = []

for idx, row in data.iterrows():
    q = row["Question"]
    must_refuse = int(row["must_refuse"]) if pd.notna(row["must_refuse"]) else 0

    # réponse générée
    ans = answer(q)
    ans_only = extract_answer_only(ans)

    # ground truth refusal (depuis le CSV)
    gt_refusal = (must_refuse == 1)

    # prédiction refusal (depuis le texte généré)
    ans_refusal = is_refusal(ans_only)

    correct = int(gt_refusal == ans_refusal)

    all_results.append({
        "ID": row.get("ID", idx),
        "Question": q,
        "must_refuse": must_refuse,
        "ans_refusal": int(ans_refusal),
        "Refusal_Correct": correct
    })

    if correct == 0:
        errors.append({
            "ID": row.get("ID", idx),
            "Question": q,
            "must_refuse": must_refuse,
            "Answer": ans_only
        })

results_df = pd.DataFrame(all_results)

refusal_accuracy_score = results_df["Refusal_Correct"].mean()

print("📊 Refusal Accuracy :", refusal_accuracy_score)


📊 Refusal Accuracy : 0.5625


# Recuperation

# Métriques de Récupération
# Precision@k
Proportion de documents récupérés qui sont pertinents.

In [ ]:
# Fonction precision@k
def precision_at_k(retrieved, relevant, k):
    """
    retrieved: List of retrieved document IDs
    relevant: Set of relevant document IDs
    k: Number of top results to consider
    """
    top_k = retrieved[:k]
    top_k = set(top_k)
    relevant_retrieved = top_k & relevant

    return len(relevant_retrieved) / k if k > 0 else 0


In [ ]:
# Convertir expected_sources en Set
import pandas as pd
import re

def parse_expected_sources(x):
    if pd.isna(x):
        return set()

    # split sur virgule OU point-virgule
    parts = re.split(r"[;,]", str(x))
    return set(p.strip() for p in parts if p.strip())


In [ ]:
# Harmoniser les noms de fichiers (IMPORTANT)
import os

def normalize_source_name(filename: str):
    """
    page_001.json  -> page_001
    page_001.html  -> page_001
    Guide candidat 2025.pdf -> Guide candidat 2025
    """
    if not filename:
        return ""
    base = os.path.basename(str(filename)).strip()
    base_no_ext = os.path.splitext(base)[0]
    return base_no_ext


In [ ]:
# Calcul Precision@k pour chaque question du CSV
import numpy as np

precision_scores = []
all_results = []

K = TOP_K  # ou mets K=4, ou K=5...

for idx, row in data.iterrows():
    q = row["Question"]
    expected = parse_expected_sources(row["expected_sources"])

    # normalisation expected
    expected_norm = set(normalize_source_name(s) for s in expected)

    # retrieval
    contexts = retrieve(q)

    retrieved_sources = [c.get("source", "") for c in contexts]
    retrieved_norm = [normalize_source_name(s) for s in retrieved_sources]

    # Precision@k
    score = precision_at_k(retrieved_norm, expected_norm, k=K)

    precision_scores.append(score)

    all_results.append({
        "ID": row.get("ID", idx),
        "Question": q,
        "expected_sources": str(row["expected_sources"]),
        "retrieved_sources": ", ".join(retrieved_sources),
        f"Precision@{K}": score
    })

print(f"📊 Precision@{K} moyenne :", np.mean(precision_scores))
print("📊 Scores :", precision_scores)


📊 Precision@4 moyenne : 0.015625
📊 Scores : [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.25, 0.0, 0.0, 0.0, 0.0, 0.25, 0.25, 0.0, 0.0]


In [ ]:
# Bonus (important) : Precision@k par catégorie
results_df = pd.DataFrame(all_results)
results_df.to_csv(f"precision_at_{K}_results.csv", index=False)
results_df.head()
results_df["category"] = data["category"].values
print(results_df.groupby("category")[f"Precision@{K}"].mean().sort_values(ascending=False))



category
career_general        0.15
career_general        0.00
concours_info         0.00
hallucination_trap    0.00
orientation           0.00
process_general       0.00
process_general       0.00
refusal_expected      0.00
sources_check         0.00
Name: Precision@4, dtype: float64


# Recall

In [ ]:
def recall_at_k(retrieved, relevant, k):
    """
    What fraction of relevant docs did we find?
    """
    top_k = set(retrieved[:k])
    relevant_retrieved = top_k & relevant

    return len(relevant_retrieved) / len(relevant) if relevant else 0


In [ ]:
import pandas as pd
import re
import os

def parse_expected_sources(x):
    if pd.isna(x):
        return set()
    parts = re.split(r"[;,]", str(x))
    return set(p.strip() for p in parts if p.strip())

def normalize_source_name(filename: str):
    if not filename:
        return ""
    base = os.path.basename(str(filename)).strip()
    return os.path.splitext(base)[0]


In [ ]:
import numpy as np

K = TOP_K  # ex: 4

recall_scores = []
all_results = []

for idx, row in data.iterrows():
    q = row["Question"]
    expected = parse_expected_sources(row["expected_sources"])

    # sources attendues normalisées
    expected_norm = set(normalize_source_name(s) for s in expected)

    # retrieval
    contexts = retrieve(q)
    retrieved_sources = [c.get("source", "") for c in contexts]
    retrieved_norm = [normalize_source_name(s) for s in retrieved_sources]

    # ✅ déduplication (doc-level)
    retrieved_unique = []
    seen = set()
    for s in retrieved_norm:
        if s not in seen:
            retrieved_unique.append(s)
            seen.add(s)

    # Recall@k
    score = recall_at_k(retrieved_unique, expected_norm, k=K)

    recall_scores.append(score)

    all_results.append({
        "ID": row.get("ID", idx),
        "Question": q,
        "expected_sources": str(row["expected_sources"]),
        "retrieved_sources": ", ".join(retrieved_sources),
        f"Recall@{K}": score
    })

print(f"📊 Recall@{K} moyen :", np.mean(recall_scores))
print("📊 Recall :", recall_scores)


📊 Recall@4 moyen : 0.046875
📊 Recall : [0.0, 0.0, 0.0, 0.0, 0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0, 0.0, 0.0, 0, 0, 0.0, 0, 0, 0, 0.0, 0, 0, 0, 0.0, 0.0, 0.0, 0, 0, 0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.25, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0]


# Recall_Precision_F@k

In [ ]:
def f1_at_k(precision, recall):
    if (precision + recall) == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


In [ ]:
#
import numpy as np
import pandas as pd
import re
import os

# ---------- Helpers ----------
def precision_at_k(retrieved, relevant, k):
    top_k = set(retrieved[:k])
    relevant_retrieved = top_k & relevant
    return len(relevant_retrieved) / k if k > 0 else 0

def recall_at_k(retrieved, relevant, k):
    top_k = set(retrieved[:k])
    relevant_retrieved = top_k & relevant
    return len(relevant_retrieved) / len(relevant) if relevant else 0

def f1_at_k(precision, recall):
    if (precision + recall) == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

def parse_expected_sources(x):
    if pd.isna(x):
        return set()
    parts = re.split(r"[;,]", str(x))
    return set(p.strip() for p in parts if p.strip())

def normalize_source_name(filename: str):
    if not filename:
        return ""
    base = os.path.basename(str(filename)).strip()
    return os.path.splitext(base)[0]


# ---------- Evaluation ----------
K = TOP_K  # ex: 4

precision_scores = []
recall_scores = []
f1_scores = []
all_results = []

for idx, row in data.iterrows():
    q = row["Question"]

    # sources attendues
    expected = parse_expected_sources(row["expected_sources"])
    expected_norm = set(normalize_source_name(s) for s in expected)

    # sources récupérées
    contexts = retrieve(q)
    retrieved_sources = [c.get("source", "") for c in contexts]
    retrieved_norm = [normalize_source_name(s) for s in retrieved_sources]

    # ✅ déduplication (document-level)
    retrieved_unique = []
    seen = set()
    for s in retrieved_norm:
        if s not in seen:
            retrieved_unique.append(s)
            seen.add(s)

    # scores
    p = precision_at_k(retrieved_unique, expected_norm, k=K)
    r = recall_at_k(retrieved_unique, expected_norm, k=K)
    f1 = f1_at_k(p, r)

    precision_scores.append(p)
    recall_scores.append(r)
    f1_scores.append(f1)

    all_results.append({
        "ID": row.get("ID", idx),
        "Question": q,
        f"Precision@{K}": p,
        f"Recall@{K}": r,
        f"F1@{K}": f1,
        "expected_sources": str(row["expected_sources"]),
        "retrieved_sources": ", ".join(retrieved_sources)
    })

results_df = pd.DataFrame(all_results)





In [ ]:
print(f"📊 Precision@{K} moyenne :", np.mean(precision_scores))
print(f"📊 Recall@{K} moyen :", np.mean(recall_scores))
print(f"📊 F1@{K} moyen :", np.mean(f1_scores))



📊 Precision@4 moyenne : 0.015625
📊 Recall@4 moyen : 0.046875
📊 F1@4 moyen : 0.021875000000000002


In [ ]:
results_df.to_csv(f"retrieval_metrics_at_{K}.csv", index=False)
results_df.head()

,ID,Question,Precision@4,Recall@4,F1@4,expected_sources,retrieved_sources
0,1,Combien de postes sont disponibles pour le con...,0.00,0.00,0.00,page_001.html,instituts_cnrs.json
1,2,Est ce que le concours Experte ou expert en in...,0.00,0.00,0.00,Guide candidat 2025.pdf,
2,3,Quelles sont les responsabilités de l'ingénieu...,0.00,0.00,0.00,page_064.html,
3,4,Quel salaire je peux prétendre en fin de carri...,0.00,0.00,0.00,Guide candidat 2025.pdf,
4,5,Est ce que le lieu du concours Experte ou expe...,0.00,0.00,0.00,nan,instituts_cnrs.json
5,6,où et quand se déroule le concours Experte ou ...,0.00,0.00,0.00,Guide candidat 2025.pdf,
6,7,Quelles sont les missions du poste ingénieur b...,0.00,0.00,0.00,page_001.html,instituts_cnrs.json
7,8,Quels sont les pré-requis pour passer le conco...,0.00,0.00,0.00,page_001.html,instituts_cnrs.json
8,9,Quel est le salaire net mensuel exact pour le ...,0.00,0.00,0.00,Guide candidat 2025.pdf,instituts_cnrs.json
9,10,Quels sont les diplômes requis pour postuler a...,0.00,0.00,0.00,page_047.html,instituts_cnrs.json
